# Phase 1 Governance Readiness Package

Notebook fix marker: `phase1_governance_readiness_v1_20260421`

Purpose: run the CLI-backed governance readiness package after the post-A4 gap review. This notebook does **not** train models, run controls, estimate calibration, estimate influence, or open headline claims.

Scientific integrity rule: completed A2/A2b, A2c, A2d, A3 and A4 smoke reviews are implementation provenance only. They must not be interpreted as decoder efficacy, comparator efficacy, privileged-transfer efficacy, or superiority evidence.

## Technical Basis And Claim Boundary

This notebook follows the repository docs and source-of-truth configs:

- `docs/01_v55_doc_code_crosswalk.md`: governance logic must live in `src/`; notebooks only orchestrate CLI, hash files, and inspect artifacts.
- `docs/02_colab_local_runbook.md`: post-A4 gap review remains non-claim; headline claims stay closed until final comparator readiness, executable controls, calibration, influence and reporting are complete.
- `configs/gate1/decision_simulation.json`: subject-level SESOI delta BA, max delta ECE, and influence ceiling are locked threshold references.
- `configs/gate2/synthetic_validation.json`: negative-control and influence threshold defaults are locked references.
- `configs/controls/*.yaml` and `configs/eval/*.yaml`: currently governance surfaces are readiness/config sources, not final claim-bearing outputs.

Expected result for the current stage: `claim_ready=false`, `headline_phase1_claim_open=false`, and `full_phase1_claim_bearing_run_allowed=false`.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.
# This cell intentionally runs repository tests before any governance artifact is written.

from pathlib import Path
import base64
import getpass
import os
import subprocess
import textwrap

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True, redact_token=True):
    display = list(map(str, cmd))
    if redact_token:
        display = ['<redacted-token-command>' if 'http.extraHeader=Authorization:' in item else item for item in display]
    print('$', ' '.join(display))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def github_auth_args():
    if not IN_COLAB:
        return []
    token = getpass.getpass('Nhập GitHub token cho repo private; Enter nếu runtime đã có quyền: ')
    if not token:
        return []
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return ['-c', f'http.extraHeader={header}']

GIT_AUTH_ARGS = github_auth_args()
if not REPO_DIR.exists():
    run(['git', *GIT_AUTH_ARGS, 'clone', REPO_URL, str(REPO_DIR)])
else:
    print(f'Repo exists: {REPO_DIR}')

run(['git', *GIT_AUTH_ARGS, 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Stop here; do not write governance readiness artifacts from an unclean codebase.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed; continuing to governance readiness checks.')

In [ ]:
# Cell 2 - Explicit source paths.
# Keep source-of-truth paths explicit. Do not silently follow latest.txt for claim-affecting steps.

from pathlib import Path

PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
POST_A4_GAP_REVIEW_RUN = DRIVE_ROOT / 'artifacts/phase1_gap_review/20260420T101100749205Z'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_governance_readiness'

EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
EXPECTED_COMPLETED_NON_CLAIM_SMOKES = [
    'A2_A2b',
    'A2c_CORAL',
    'A2d_riemannian',
    'A3_distillation',
    'A4_privileged',
]

print('Prereg bundle:', PREREG_BUNDLE)
print('Post-A4 gap review:', POST_A4_GAP_REVIEW_RUN)
print('Governance output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - Small JSON/hash helpers used only for orchestration review.

import hashlib
import json
from pathlib import Path


def read_json(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def latest_run(root: Path) -> Path:
    pointer = Path(root) / 'latest.txt'
    if not pointer.exists():
        raise FileNotFoundError(pointer)
    return Path(pointer.read_text(encoding='utf-8').strip())

In [ ]:
# Cell 4 - Validate prereg identity and post-A4 gap-review boundary before running readiness.

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

gap_summary = read_json(POST_A4_GAP_REVIEW_RUN / 'phase1_comparator_suite_gap_review_summary.json')
gap_blockers = read_json(POST_A4_GAP_REVIEW_RUN / 'claim_readiness_blockers.json')
print('Gap review status:', gap_summary.get('status'))
print('Gap review claim_ready:', gap_summary.get('claim_ready'))
print('Gap review completed non-claim smoke reviews:', gap_summary.get('completed_non_claim_smoke_reviews'))
print('Gap review blockers:', gap_summary.get('blockers'))
assert gap_summary.get('status') == 'phase1_comparator_suite_gap_review_complete'
assert gap_summary.get('claim_ready') is False
assert gap_summary.get('headline_phase1_claim_open') is False
assert gap_blockers.get('claim_state', {}).get('full_phase1_claim_bearing_run_allowed') is False
for item in EXPECTED_COMPLETED_NON_CLAIM_SMOKES:
    assert item in gap_summary.get('completed_non_claim_smoke_reviews', []), f'Missing smoke review: {item}'

In [ ]:
# Cell 5 - Hash-link governance source modules and config surfaces.
# These hashes document what code/config generated this readiness package.

HASH_TARGETS = [
    'src/phase1/controls.py',
    'src/phase1/calibration.py',
    'src/phase1/influence.py',
    'src/phase1/claim_state.py',
    'src/cli.py',
    'tests/unit/test_phase1_governance_readiness.py',
    'configs/controls/control_suite_spec.yaml',
    'configs/controls/nuisance_block_spec.yaml',
    'configs/eval/claim_mapping.yaml',
    'configs/eval/metrics.yaml',
    'configs/eval/inference_defaults.yaml',
    'configs/gate1/decision_simulation.json',
    'configs/gate2/synthetic_validation.json',
    'notebooks/15_colab_phase1_governance_readiness.ipynb',
]

hash_manifest = []
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    exists = path.exists()
    hash_manifest.append({'path': rel, 'exists': exists, 'sha256': sha256_file(path) if exists else None})

print(json.dumps(hash_manifest, indent=2))
assert all(item['exists'] for item in hash_manifest), 'All governance source/config/notebook files must exist.'

## Governance Readiness Artifact Contract

The CLI command below must write these files under a timestamped run directory:

- `phase1_governance_readiness_inputs.json`
- `phase1_governance_readiness_summary.json`
- `phase1_governance_readiness_report.md`
- `phase1_control_suite_status.json`
- `phase1_calibration_package_status.json`
- `phase1_influence_status.json`
- `phase1_reporting_readiness.json`
- `phase1_claim_state.json`

For the current repository state, the expected interpretation is blocked/non-claim. Missing final controls, calibration, influence and reporting are blockers, not values to infer or fill manually.

In [ ]:
# Cell 6 - Run the CLI-backed governance readiness package.
# This command does not train models and does not compute final controls/calibration/influence.

cmd = [
    'python', '-m', 'src.cli', 'phase1_governance_readiness',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--gap-review-run', str(POST_A4_GAP_REVIEW_RUN),
    '--output-root', str(OUTPUT_ROOT),
]
run(cmd, cwd=REPO_DIR)
print('Governance readiness command completed. Review generated artifacts before any further claim-bearing implementation.')

In [ ]:
# Cell 7 - Verify artifact contract and print closeout summary.

run_dir = latest_run(OUTPUT_ROOT)
print('Latest governance output:', run_dir)
required_files = [
    'phase1_governance_readiness_inputs.json',
    'phase1_governance_readiness_summary.json',
    'phase1_governance_readiness_report.md',
    'phase1_control_suite_status.json',
    'phase1_calibration_package_status.json',
    'phase1_influence_status.json',
    'phase1_reporting_readiness.json',
    'phase1_claim_state.json',
]
for name in required_files:
    exists = (run_dir / name).exists()
    print(f'{name} exists = {exists}')
    assert exists, f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_governance_readiness_summary.json')
claim_state = read_json(run_dir / 'phase1_claim_state.json')
controls = read_json(run_dir / 'phase1_control_suite_status.json')
calibration = read_json(run_dir / 'phase1_calibration_package_status.json')
influence = read_json(run_dir / 'phase1_influence_status.json')
reporting = read_json(run_dir / 'phase1_reporting_readiness.json')

print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'full_phase1_claim_bearing_run_allowed': summary.get('full_phase1_claim_bearing_run_allowed'),
    'completed_non_claim_smoke_reviews': summary.get('completed_non_claim_smoke_reviews'),
    'governance_surfaces': summary.get('governance_surfaces'),
    'control_status': controls.get('status'),
    'calibration_status': calibration.get('status'),
    'influence_status': influence.get('status'),
    'reporting_status': reporting.get('status'),
    'blockers': summary.get('blockers'),
}, indent=2))

assert summary.get('status') == 'phase1_governance_readiness_blocked'
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert claim_state.get('status') == 'phase1_claim_state_blocked'
assert claim_state.get('claim_ready') is False
assert controls.get('claim_evaluable') is False
assert calibration.get('claim_evaluable') is False
assert influence.get('claim_evaluable') is False
assert reporting.get('claim_evaluable') is False
print('\nNOT OK TO CLAIM: This governance readiness package does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')

## Next Engineering Boundary

Allowed next step: implement the final executable control/calibration/influence/reporting package under prereg/revision discipline, with tests that verify final artifacts exist and thresholds are read from locked configs/artifacts.

Not allowed: interpreting any smoke BA/ECE/Brier value as efficacy evidence, filling missing control/calibration/influence outputs by hand, or opening headline Phase 1 claims from this readiness package.